# VoiceHub data preparation: TTS, ASR, and VAD

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kadirnar/voicehub/blob/main/notebooks/data_preparation.ipynb)

This notebook builds auditable source records, inspects architecture-specific contracts, normalizes common aliases, creates group-disjoint splits, and records stable resume fingerprints. Its core cells require no model download or tensor framework.

Keep consent, license, provenance, speaker, session, language, and source checksums in the source manifest even when a model adapter does not consume every field.

## 0. Install the data and training surface

Use the `training` extra when model-owned preprocessing will create tokens, codec targets, acoustic features, or aligned labels. The setup cell installs it only when VoiceHub is not already importable.

In [ ]:
import importlib.util
import subprocess
import sys

INSTALL_VOICEHUB = importlib.util.find_spec("voicehub") is None
VOICEHUB_REVISION = "main"  # Prefer a release tag or full commit SHA.

if INSTALL_VOICEHUB:
    package = (
        "voicehub[training] @ "
        "git+https://github.com/kadirnar/voicehub.git@"
        f"{VOICEHUB_REVISION}"
    )
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "--upgrade",
        package,
    ])


In [ ]:
from pathlib import Path

TTS_MODEL_TYPE = "dia"
ASR_MODEL_TYPE = "asr_wav2vec2"
DATA_ROOT = Path("data")
OUTPUT_ROOT = DATA_ROOT / "prepared"

WRITE_EXAMPLE_MANIFESTS = False
RUN_AUDIO_VALIDATION = False
RUN_MODEL_PREPARATION = False


## 1. Inspect contracts before creating data

A contract declares accepted source and prepared variants for one registered model. `integrated-raw` means the model adapter can own preprocessing from a source record; `preprocessed` means the caller must supply the exact model-shaped variant.

In [ ]:
from voicehub import (
    ASRDataArchitecture,
    TTSDataArchitecture,
    get_asr_dataset_spec,
    get_tts_dataset_spec,
)


def describe_contract(contract):
    return {
        "model_type": contract.model_type,
        "architecture": contract.architecture.value,
        "readiness": (
            None
            if contract.readiness is None
            else contract.readiness.value
        ),
        "sample_rate": contract.sample_rate,
        "raw_variants": tuple(v.name for v in contract.raw_variants),
        "prepared_variants": tuple(
            v.name for v in contract.preprocessed_variants
        ),
    }


tts_architecture_contracts = {
    architecture.value: get_tts_dataset_spec(architecture=architecture)
    for architecture in TTSDataArchitecture
}
asr_architecture_contracts = {
    architecture.value: get_asr_dataset_spec(architecture=architecture)
    for architecture in ASRDataArchitecture
}

print("TTS architecture contracts:")
for name, contract in tts_architecture_contracts.items():
    print(" ", name, describe_contract(contract))
print("ASR architecture contracts:")
for name, contract in asr_architecture_contracts.items():
    print(" ", name, describe_contract(contract))

model_contract_examples = {
    "tts_codec_lm": get_tts_dataset_spec("conversationtts"),
    "tts_sequence": get_tts_dataset_spec("dia"),
    "tts_diffusion": get_tts_dataset_spec("f5tts"),
    "tts_vits": get_tts_dataset_spec("vits"),
    "asr_ctc": get_asr_dataset_spec("asr_wav2vec2"),
    "asr_seq2seq": get_asr_dataset_spec("asr_whisper"),
    "asr_transducer": get_asr_dataset_spec("asr_nemotron"),
}

for name, contract in model_contract_examples.items():
    print(name, describe_contract(contract))


## 2. Create source records in memory

Paths may remain unresolved while designing a corpus. Set `validate_files=True` only after the files have been materialized. The examples include two sessions or speakers so a grouped split can be verified.

In [ ]:
tts_records = [
    {
        "id": "tts-s01-0001",
        "text": "The first authorized training utterance.",
        "audio": "tts/s01/0001.wav",
        "speaker_id": "speaker-01",
        "session_id": "session-01",
        "language": "en",
        "consent": True,
        "license": "owned",
    },
    {
        "id": "tts-s01-0002",
        "text": "A second recording from the same session.",
        "audio": "tts/s01/0002.wav",
        "speaker_id": "speaker-01",
        "session_id": "session-01",
        "language": "en",
        "consent": True,
        "license": "owned",
    },
    {
        "id": "tts-s02-0001",
        "text": "Validation must use a disjoint recording session.",
        "audio": "tts/s02/0001.wav",
        "speaker_id": "speaker-01",
        "session_id": "session-02",
        "language": "en",
        "consent": True,
        "license": "owned",
    },
    {
        "id": "tts-s02-0002",
        "text": "Keep provenance beside every transcript.",
        "audio": "tts/s02/0002.wav",
        "speaker_id": "speaker-01",
        "session_id": "session-02",
        "language": "en",
        "consent": True,
        "license": "owned",
    },
]

asr_records = [
    {
        "id": "asr-a-0001",
        "audio_filepath": "asr/a/0001.wav",
        "transcript": "The first verified transcript.",
        "speaker_id": "speaker-a",
        "language": "en",
    },
    {
        "id": "asr-a-0002",
        "audio_filepath": "asr/a/0002.wav",
        "transcript": "Aliases normalize at the dataset boundary.",
        "speaker_id": "speaker-a",
        "language": "en",
    },
    {
        "id": "asr-b-0001",
        "audio_filepath": "asr/b/0001.wav",
        "transcript": "The held-out speaker remains disjoint.",
        "speaker_id": "speaker-b",
        "language": "en",
    },
    {
        "id": "asr-b-0002",
        "audio_filepath": "asr/b/0002.wav",
        "transcript": "Source metadata remains auditable.",
        "speaker_id": "speaker-b",
        "language": "en",
    },
]

vad_records = [
    {
        "id": "meeting-001",
        "audio": "vad/meeting-001.wav",
        "duration": 8.0,
        "segments": [
            {"start": 0.5, "end": 2.0, "label": "speech"},
            {"start": 3.25, "end": 6.5, "label": "speech"},
        ],
    },
    {
        "id": "meeting-002",
        "audio": "vad/meeting-002.wav",
        "duration": 5.0,
        "segments": [
            {"start": 1.0, "end": 4.0, "label": "speech"},
        ],
    },
]


## 3. Normalize and validate portable datasets

`TTSDataset` and `ASRDataset` validate model-specific record variants without decoding audio. Common aliases such as `audio_filepath` and `transcript` become canonical `audio` and `text` fields.

In [ ]:
from voicehub import ASRDataset, SpeechDataset, TTSDataset

tts_source = TTSDataset(
    tts_records,
    model_type=TTS_MODEL_TYPE,
    root=DATA_ROOT,
    validate_files=False,
)
asr_source = ASRDataset(
    asr_records,
    model_type=ASR_MODEL_TYPE,
    root=DATA_ROOT,
    validate_files=False,
)
vad_source = SpeechDataset(
    vad_records,
    required_fields=("audio", "segments"),
)

print(tts_source.architecture.value, tts_source.variant_names)
print(asr_source.architecture.value, asr_source.variant_names)
print(asr_source[0]["audio"], asr_source[0]["text"])
print("VAD records:", len(vad_source))


## 4. Split on the strongest leakage boundary

Adjacent clips often share room tone, microphones, speakers, and source recordings. Split whole sessions or speakers before windowing and augmentation.

In [ ]:
tts_train, tts_validation = tts_source.train_test_split(
    validation_fraction=0.5,
    seed=42,
    group_by="session_id",
)
asr_train, asr_validation = asr_source.train_test_split(
    validation_fraction=0.5,
    seed=42,
    group_by="speaker_id",
)

tts_train_sessions = {row["session_id"] for row in tts_train}
tts_validation_sessions = {row["session_id"] for row in tts_validation}
asr_train_speakers = {row["speaker_id"] for row in asr_train}
asr_validation_speakers = {row["speaker_id"] for row in asr_validation}

assert tts_train_sessions.isdisjoint(tts_validation_sessions)
assert asr_train_speakers.isdisjoint(asr_validation_speakers)

tts_fingerprint = tts_train.resume_fingerprint()
asr_fingerprint = asr_train.resume_fingerprint()
print("TTS split:", len(tts_train), len(tts_validation), tts_fingerprint)
print("ASR split:", len(asr_train), len(asr_validation), asr_fingerprint)


## 5. Validate VAD time intervals

VAD annotations remain on the original recording timebase. Validate ordering, overlap, bounds, and labels before converting intervals into provider-specific frame targets.

In [ ]:
def validate_vad_intervals(record):
    duration = float(record["duration"])
    previous_end = 0.0
    for segment in record["segments"]:
        start = float(segment["start"])
        end = float(segment["end"])
        if start < previous_end or end <= start or end > duration:
            raise ValueError(
                f"Invalid VAD interval {start}..{end} for duration {duration}"
            )
        if not str(segment.get("label", "")).strip():
            raise ValueError("Every VAD interval requires a label")
        previous_end = end


for vad_record in vad_source:
    validate_vad_intervals(vad_record)
print("Validated", len(vad_source), "VAD records")


## 6. Persist immutable split manifests

JSON Lines keeps every record independently inspectable. Store the split seed, preprocessing revision, VoiceHub revision, source checksums, and dataset fingerprints alongside the files.

In [ ]:
if WRITE_EXAMPLE_MANIFESTS:
    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    tts_train.to_jsonl(OUTPUT_ROOT / "tts-train.jsonl")
    tts_validation.to_jsonl(OUTPUT_ROOT / "tts-validation.jsonl")
    asr_train.to_jsonl(OUTPUT_ROOT / "asr-train.jsonl")
    asr_validation.to_jsonl(OUTPUT_ROOT / "asr-validation.jsonl")
    print("Wrote manifests to", OUTPUT_ROOT)
else:
    print("Manifest writing is disabled; set WRITE_EXAMPLE_MANIFESTS=True.")


## 7. Import common corpus layouts

Use `TTSDataset.from_manifest()` for JSON/JSONL/CSV/TSV, `TTSDataset.from_ljspeech()` for `metadata.csv` plus `wavs/`, `ASRDataset.from_audio_folder()` for same-stem WAV/transcript sidecars, and `ASRDataset.from_kaldi()` for materialized `wav.scp` plus `text`. Kaldi shell pipelines are rejected; materialize them as audio files first.

Set `validate_files=True` only after paths resolve on the machine preparing the corpus.

## 8. Materialize and validate audio

`load_audio()` produces finite mono float32 audio and can resample explicitly. Keep the original file and checksum; normalized audio is a versioned derived artifact.

In [ ]:
if RUN_AUDIO_VALIDATION:
    from voicehub import load_audio

    first_path = Path(tts_source[0]["audio"])
    if not first_path.is_file():
        raise FileNotFoundError(first_path)
    audio = load_audio(first_path, target_sampling_rate=44_100)
    print(audio.waveform.shape, audio.sampling_rate, audio.duration)
else:
    print("Audio validation is disabled until real files are available.")


## 9. Let the selected model own final preparation

Portable datasets stop before tokenization, codec extraction, feature computation, alignment, and model-specific label construction. A loaded wrapper turns source records into its exact training dataset. Inspect one collated batch before launching a run.

In [ ]:
if RUN_MODEL_PREPARATION:
    from voicehub import AutoModelForTextToSpeech

    training_model = AutoModelForTextToSpeech.from_pretrained(
        "nari-labs/Dia-1.6B-0626",
        model_type=TTS_MODEL_TYPE,
        device="cuda",
        lazy_load=True,
    )
    prepared_train = training_model.create_training_dataset(tts_train)
    features = [
        prepared_train[index]
        for index in range(min(2, len(prepared_train)))
    ]
    batch = prepared_train.collate_fn(features)
    for name, value in batch.items():
        print(name, getattr(value, "shape", type(value).__name__))
else:
    print("Model-owned preparation is disabled.")


## Next steps

- Read the [TTS data guide](https://kadirnar.github.io/voicehub/guides/data-preparation/) and [ASR/VAD data guide](https://kadirnar.github.io/voicehub/guides/speech-data/).
- Compare the selected contract with the [training support matrix](https://kadirnar.github.io/voicehub/models/training-support/).
- Continue with the [training notebook](training.ipynb) for VITS, codec/LLM, diffusion/flow, and ASR recipes.